# K-Fold Training v3 (Parallel T4x2)

This notebook runs K-fold training with 2-GPU DataParallel + AMP and logs fold error snapshots every 250 steps.


In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:128")

REPO_URL = "https://github.com/mruniverse8/kaggle-experiments-.git"
REPO_DIR = Path("/kaggle/working/kaggle-experiments-")
BRANCH = "v3_parallel"

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)
subprocess.run(["git", "fetch", "--all"], check=True)
subprocess.run(["git", "checkout", BRANCH], check=True)
subprocess.run(["git", "reset", "--hard", f"origin/{BRANCH}"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

current_branch = subprocess.check_output(["git", "rev-parse", "--abbrev-ref", "HEAD"]).decode().strip()
current_commit = subprocess.check_output(["git", "rev-parse", "--short", "HEAD"]).decode().strip()
print("Git branch:", current_branch)
print("Git commit:", current_commit)
print("Repo ready at:", REPO_DIR)


In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from contextlib import nullcontext
from sklearn.model_selection import StratifiedKFold
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, get_linear_schedule_with_warmup

import sys
sys.path.insert(0, str(Path("src").resolve()))

from dz2_causal.dataset import (
    TweetExtractionCausalDataset,
    causal_collate_fn,
    register_special_tokens,
)
from dz2_causal.losses import compute_total_loss
from dz2_causal.modeling import CausalExtractionModel
from dz2_causal.eval_utils import evaluate_dataframe_jaccard

CFG_PATH = os.environ.get("CFG_PATH", "config/kaggle_train_kfold_v3_parallel_t4x2.json")
print("Using config:", CFG_PATH)
CFG = json.loads(Path(CFG_PATH).read_text())
CFG

print(
    "use_data_parallel:", CFG.get("use_data_parallel", False),
    "| use_amp:", CFG.get("use_amp", not bool(CFG.get("no_amp", False))),
    "| amp_dtype:", CFG.get("amp_dtype", "auto"),
    "| model:", CFG["model_name"],
)

print("CUDA available:", torch.cuda.is_available(), "| device_count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for gpu_id in range(torch.cuda.device_count()):
        free_b, total_b = torch.cuda.mem_get_info(gpu_id)
        print(
            f"cuda:{gpu_id} startup free={free_b/(1024**3):.2f}GiB "
            f"total={total_b/(1024**3):.2f}GiB"
        )


In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG["seed"])

df = pd.read_csv(CFG["train_csv"]).dropna(subset=["text", "selected_text"]).reset_index(drop=True)
print("Train rows:", len(df))

splits = list(
    StratifiedKFold(
        n_splits=CFG["n_splits"],
        shuffle=True,
        random_state=CFG["seed"],
    ).split(df, df["sentiment"])
)

OUTPUT_DIR = Path(CFG["output_dir"])
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print("Output dir:", OUTPUT_DIR)


In [ ]:
def make_records(frame: pd.DataFrame):
    records = frame[["text", "sentiment", "selected_text"]].to_dict("records")
    for r in records:
        r["prompt"] = CFG["prompt_text"]
    return records


def unwrap_model(model):
    return model.module if isinstance(model, torch.nn.DataParallel) else model


def align_head_dtypes_with_lm(model, device):
    base_model = unwrap_model(model)
    target_dtype = base_model.lm.get_input_embeddings().weight.dtype
    base_model.start_head.to(device=device, dtype=target_dtype)
    base_model.end_head.to(device=device, dtype=target_dtype)
    base_model.select_head.to(device=device, dtype=target_dtype)
    return target_dtype


def compute_grad_norm_l2(model) -> float:
    base_model = unwrap_model(model)
    total = 0.0
    for p in base_model.parameters():
        if p.grad is None:
            continue
        g = p.grad.detach()
        total += float(g.float().pow(2).sum().item())
    return float(total ** 0.5)


def reduce_loss_tensor(loss_val: torch.Tensor) -> torch.Tensor:
    if torch.is_tensor(loss_val) and loss_val.ndim > 0:
        return loss_val.mean()
    return loss_val


def loss_to_float(loss_val: torch.Tensor) -> float:
    reduced = reduce_loss_tensor(loss_val)
    return float(reduced.detach().item())


def resolve_parallel_devices(cfg):
    if not torch.cuda.is_available():
        return []

    available_ids = list(range(torch.cuda.device_count()))
    requested_ids = cfg.get("parallel_gpu_ids", available_ids)
    requested_ids = [int(i) for i in requested_ids if int(i) in available_ids]
    if not requested_ids:
        requested_ids = available_ids

    if cfg.get("use_data_parallel", False) and len(requested_ids) >= 2:
        return requested_ids
    return [requested_ids[0]]


def resolve_amp_settings(cfg):
    use_amp = bool(cfg.get("use_amp", not bool(cfg.get("no_amp", False))))
    if not torch.cuda.is_available() or not use_amp:
        return False, None, False

    amp_pref = str(cfg.get("amp_dtype", "fp16")).lower()
    if amp_pref == "bf16":
        if not torch.cuda.is_bf16_supported():
            raise ValueError("amp_dtype='bf16' requested but GPU does not support bf16.")
        amp_dtype = torch.bfloat16
    elif amp_pref == "fp16":
        amp_dtype = torch.float16
    elif amp_pref == "auto":
        amp_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    else:
        raise ValueError("amp_dtype must be one of: fp16, bf16, auto")

    use_grad_scaler = amp_dtype == torch.float16
    return True, amp_dtype, use_grad_scaler


def build_memory_snapshot(device_ids, min_free_mem_gb: float):
    if not torch.cuda.is_available() or not device_ids:
        return "cpu", []

    parts = []
    low_free_devices = []
    for did in device_ids:
        free_b, total_b = torch.cuda.mem_get_info(did)
        alloc_b = torch.cuda.memory_allocated(did)
        reserved_b = torch.cuda.memory_reserved(did)

        free_gb = free_b / (1024 ** 3)
        total_gb = total_b / (1024 ** 3)
        alloc_gb = alloc_b / (1024 ** 3)
        reserved_gb = reserved_b / (1024 ** 3)

        if free_gb < min_free_mem_gb:
            low_free_devices.append(did)

        parts.append(
            f"cuda:{did} alloc={alloc_gb:.2f}G reserved={reserved_gb:.2f}G free={free_gb:.2f}/{total_gb:.2f}G"
        )

    return " | ".join(parts), low_free_devices


def run_smoke_step(model, loader, optimizer, scheduler, device, use_amp=False, amp_dtype=None, use_grad_scaler=False, scaler=None):
    smoke_batch = next(iter(loader))
    for k, v in smoke_batch.items():
        if torch.is_tensor(v):
            smoke_batch[k] = v.to(device)

    amp_context = (
        torch.amp.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp)
        if device.type == "cuda"
        else nullcontext()
    )

    with amp_context:
        out = model(
            input_ids=smoke_batch["input_ids"],
            attention_mask=smoke_batch["attention_mask"],
            labels=smoke_batch["labels"],
            compute_span_logits=False,
        )
        losses = compute_total_loss(
            outputs=out,
            batch=smoke_batch,
            lambda_kl=0.0,
            lambda_select=0.0,
        )
        loss = reduce_loss_tensor(losses["loss"])
        ce_loss = reduce_loss_tensor(losses["ce_loss"])

    print(f"[smoke] loss={loss.item():.4f} ce={ce_loss.item():.4f}")

    if use_grad_scaler:
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
    else:
        loss.backward()
        optimizer.step()

    optimizer.zero_grad(set_to_none=True)
    scheduler.step()


def train_one_fold(fold_id: int, train_idx, val_idx):
    print(f"\n===== Fold {fold_id} =====")

    tokenizer = AutoTokenizer.from_pretrained(
        CFG["model_name"],
        use_fast=True,
        trust_remote_code=CFG.get("trust_remote_code", False),
    )

    base_model = CausalExtractionModel(
        model_name=CFG["model_name"],
        trust_remote_code=CFG.get("trust_remote_code", False),
    )
    if CFG.get("gradient_checkpointing", False):
        base_model.lm.gradient_checkpointing_enable()
        if hasattr(base_model.lm, "config"):
            base_model.lm.config.use_cache = False
        print("Gradient checkpointing: enabled")
    register_special_tokens(tokenizer, model=base_model.lm)

    train_frame = df.iloc[train_idx].reset_index(drop=True)
    if CFG.get("filter_long_samples", False) and "seq_len" in train_frame.columns:
        before_count = len(train_frame)
        train_frame = train_frame[train_frame["seq_len"] <= int(CFG["max_len"])].reset_index(drop=True)
        dropped_count = before_count - len(train_frame)
        if dropped_count > 0:
            print(f"[fold {fold_id}] dropped {dropped_count} over-length samples (max_len={CFG['max_len']})")

    if len(train_frame) == 0:
        raise ValueError(f"Fold {fold_id} has no training rows after length filtering. Increase max_len.")

    train_ds = TweetExtractionCausalDataset(
        records=make_records(train_frame),
        tokenizer=tokenizer,
        prompt_text=CFG["prompt_text"],
        max_len=CFG["max_len"],
        soft_alpha=CFG["soft_alpha"],
    )

    train_loader = DataLoader(
        train_ds,
        batch_size=CFG["batch_size"],
        shuffle=True,
        num_workers=CFG["num_workers"],
        collate_fn=lambda b: causal_collate_fn(b, tokenizer.pad_token_id),
    )

    device_ids = resolve_parallel_devices(CFG)
    if torch.cuda.is_available():
        device = torch.device(f"cuda:{device_ids[0]}")
    else:
        device = torch.device("cpu")

    base_model.to(device)
    model = base_model

    use_dp = bool(CFG.get("use_data_parallel", False))
    if torch.cuda.is_available() and use_dp and len(device_ids) > 1:
        model = torch.nn.DataParallel(base_model, device_ids=device_ids, output_device=device_ids[0])
        print(
            f"DataParallel: enabled on GPUs {device_ids} | global_batch_size={CFG['batch_size']} "
            f"approx_per_gpu={max(1, CFG['batch_size'] // len(device_ids))}"
        )
    else:
        print(f"DataParallel: disabled | device={device}")

    lm_dtype = align_head_dtypes_with_lm(base_model, device)
    print(f"Aligned heads to LM dtype: {lm_dtype}")

    use_amp, amp_dtype, use_grad_scaler = resolve_amp_settings(CFG)
    scaler = torch.amp.GradScaler("cuda", enabled=(use_grad_scaler and device.type == "cuda"))
    print(f"AMP: enabled={use_amp} dtype={amp_dtype} grad_scaler={use_grad_scaler}")

    optimizer = torch.optim.AdamW(
        base_model.parameters(),
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"],
    )

    total_steps = max(1, CFG["epochs"] * len(train_loader))
    warmup_steps = int(CFG["warmup_ratio"] * total_steps)
    scheduler = get_linear_schedule_with_warmup(optimizer, warmup_steps, total_steps)
    optimizer.zero_grad(set_to_none=True)

    min_free_mem_gb = float(CFG.get("min_free_mem_gb", 1.0))
    mem_status, low_devices = build_memory_snapshot(device_ids, min_free_mem_gb)
    if torch.cuda.is_available():
        print(f"[fold {fold_id}] startup memory | {mem_status}")
        if low_devices:
            print(f"[fold {fold_id}] warning: free memory below {min_free_mem_gb:.2f}GB on GPUs {low_devices}")

    ckpt_root = OUTPUT_DIR / "checkpoints" / f"fold{fold_id}"
    ckpt_root.mkdir(parents=True, exist_ok=True)

    def save_ckpt(tag: str, epoch_id: int, step_id: int, gstep: int):
        ckpt = {
            "model_state_dict": base_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "model_name": CFG["model_name"],
            "config": CFG,
            "fold": int(fold_id),
            "epoch": int(epoch_id),
            "step": int(step_id),
            "global_step": int(gstep),
        }
        ckpt_path = ckpt_root / f"{tag}.pt"
        torch.save(ckpt, ckpt_path)
        latest_path = ckpt_root / "latest.pt"
        torch.save(ckpt, latest_path)
        return ckpt_path

    if CFG.get("run_smoke_test", True):
        run_smoke_step(
            model,
            train_loader,
            optimizer,
            scheduler,
            device,
            use_amp=use_amp,
            amp_dtype=amp_dtype,
            use_grad_scaler=use_grad_scaler,
            scaler=scaler,
        )

    history = []
    global_step = 0

    model.train()
    for epoch in range(CFG["epochs"]):
        ce_only = epoch < CFG["ce_only_epochs"]
        lambda_kl = 0.0 if ce_only else CFG["lambda_kl"]
        lambda_select = 0.0 if ce_only else CFG["lambda_select"]
        need_span = (lambda_kl > 0.0) or (lambda_select > 0.0)

        running = {"loss": 0.0, "ce": 0.0, "kl": 0.0, "sel": 0.0}
        grad_norm_sum = 0.0
        grad_norm_count = 0
        for step, batch in enumerate(train_loader):
            for k, v in batch.items():
                if torch.is_tensor(v):
                    batch[k] = v.to(device)

            amp_context = (
                torch.amp.autocast(device_type="cuda", dtype=amp_dtype, enabled=use_amp)
                if device.type == "cuda"
                else nullcontext()
            )

            with amp_context:
                out = model(
                    input_ids=batch["input_ids"],
                    attention_mask=batch["attention_mask"],
                    labels=batch["labels"],
                    compute_span_logits=need_span,
                )
                losses = compute_total_loss(
                    out,
                    batch,
                    lambda_kl=lambda_kl,
                    lambda_select=lambda_select,
                )
                full_loss = reduce_loss_tensor(losses["loss"])
                loss = full_loss / CFG["grad_accum_steps"]

            if use_grad_scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if (step + 1) % CFG["grad_accum_steps"] == 0:
                grad_norm = compute_grad_norm_l2(base_model)
                grad_norm_sum += grad_norm
                grad_norm_count += 1

                if use_grad_scaler:
                    scaler.step(optimizer)
                    scaler.update()
                else:
                    optimizer.step()
                optimizer.zero_grad(set_to_none=True)
                scheduler.step()
                global_step += 1

            running["loss"] += loss_to_float(losses["loss"])
            running["ce"] += loss_to_float(losses["ce_loss"])
            running["kl"] += loss_to_float(losses["kl_loss"])
            running["sel"] += loss_to_float(losses["select_loss"])

            if (step + 1) % CFG["log_every"] == 0:
                denom = float(CFG["log_every"])
                loss_row = {
                    "fold": int(fold_id),
                    "epoch": int(epoch + 1),
                    "step": int(step + 1),
                    "global_step": int(global_step),
                    "loss": float(running["loss"] / denom),
                    "ce": float(running["ce"] / denom),
                    "kl": float(running["kl"] / denom),
                    "sel": float(running["sel"] / denom),
                    "grad_norm": float(grad_norm_sum / max(1, grad_norm_count)),
                }
                history.append(loss_row)

                mem_suffix = ""
                if torch.cuda.is_available():
                    mem_status, low_devices = build_memory_snapshot(device_ids, min_free_mem_gb)
                    mem_suffix = f" | {mem_status}"
                    if low_devices:
                        mem_suffix += f" | warning_free_lt_{min_free_mem_gb:.2f}GB_on={low_devices}"

                print(
                    f"[fold {fold_id}] epoch={epoch + 1} step={step + 1}/{len(train_loader)} "
                    f"fold_error={loss_row['loss']:.4f} ce={loss_row['ce']:.4f} "
                    f"kl={loss_row['kl']:.4f} sel={loss_row['sel']:.4f} "
                    f"grad_norm={loss_row['grad_norm']:.4f}{mem_suffix}"
                )

                running = {"loss": 0.0, "ce": 0.0, "kl": 0.0, "sel": 0.0}
                grad_norm_sum = 0.0
                grad_norm_count = 0

        grad_norm_sum = 0.0
        grad_norm_count = 0

        epoch_ckpt_path = save_ckpt(
            tag=f"epoch_{epoch + 1}",
            epoch_id=epoch + 1,
            step_id=len(train_loader),
            gstep=global_step,
        )
        print(f"Saved epoch checkpoint: {epoch_ckpt_path}")

    val_df = df.iloc[val_idx].reset_index(drop=True)
    eval_out = evaluate_dataframe_jaccard(
        df=val_df,
        model=base_model,
        tokenizer=tokenizer,
        prompt_text=CFG["prompt_text"],
        device=device,
        max_new_tokens=CFG["max_new_tokens"],
    )

    ckpt_path = OUTPUT_DIR / f"model_fold{fold_id}.pt"
    torch.save(
        {
            "model_state_dict": base_model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "scheduler_state_dict": scheduler.state_dict(),
            "model_name": CFG["model_name"],
            "fold": fold_id,
            "val_jaccard": eval_out["mean_jaccard"],
            "config": CFG,
            "global_step": int(global_step),
        },
        ckpt_path,
    )

    print(f"Fold {fold_id} val_jaccard={eval_out['mean_jaccard']:.4f} | val_error={1.0 - eval_out['mean_jaccard']:.4f} | saved: {ckpt_path}")
    return {
        "fold": fold_id,
        "val_jaccard": eval_out["mean_jaccard"],
        "checkpoint": str(ckpt_path),
        "checkpoint_dir": str(ckpt_root),
        "history": history,
    }


## Sequence-Length Analysis and Max Length Policy

Analyze tokenized lengths, then either keep configured `max_len` (small-memory mode) or auto-adjust it via config flags.


In [ ]:
# Analyze sequence lengths and apply max_len policy from config.
import dz2_causal.dataset as dataset_utils

probe_tokenizer = AutoTokenizer.from_pretrained(
    CFG["model_name"],
    use_fast=True,
    trust_remote_code=CFG.get("trust_remote_code", False),
)
register_special_tokens(probe_tokenizer)

extra_tokens = int(probe_tokenizer.bos_token_id is not None) + int(probe_tokenizer.eos_token_id is not None)
sample_lengths = []

for row in df[["text", "sentiment", "selected_text"]].to_dict("records"):
    prompt = CFG["prompt_text"]
    tweet = dataset_utils.normalize_kaggle_span_text(row["text"])
    sentiment = dataset_utils.normalize_text(row["sentiment"])
    selected_text = dataset_utils.normalize_kaggle_span_text(row["selected_text"])

    prefix_text = (
        f"{dataset_utils.SPECIAL_TOKENS.prompt_open} {prompt} {dataset_utils.SPECIAL_TOKENS.prompt_close} "
        f"{dataset_utils.SPECIAL_TOKENS.tweet_open}{tweet} {dataset_utils.SPECIAL_TOKENS.tweet_close} "
        f"{dataset_utils.SPECIAL_TOKENS.sentiment_open} {sentiment} {dataset_utils.SPECIAL_TOKENS.sentiment_close} "
        f"{dataset_utils.SPECIAL_TOKENS.answer_open}"
    )
    full_text = f"{prefix_text}{selected_text} {dataset_utils.SPECIAL_TOKENS.answer_close}"
    tokenized = probe_tokenizer(full_text, add_special_tokens=False)["input_ids"]
    sample_lengths.append(len(tokenized) + extra_tokens)

df["seq_len"] = sample_lengths
computed_max_len = int(np.max(sample_lengths))
p95_len = int(np.percentile(sample_lengths, 95))
p99_len = int(np.percentile(sample_lengths, 99))
old_max_len = int(CFG["max_len"])

if CFG.get("auto_set_max_len", False):
    target_max_len = computed_max_len
    auto_cap = CFG.get("auto_max_len_cap")
    if auto_cap is not None:
        target_max_len = min(target_max_len, int(auto_cap))
    CFG["max_len"] = int(target_max_len)
    print(f"Auto max_len enabled: config {old_max_len} -> {CFG['max_len']}")
else:
    print(f"Auto max_len disabled: using configured max_len={old_max_len}")

too_long_count = int((df["seq_len"] > int(CFG["max_len"])).sum())
print(f"Length stats | max={computed_max_len} p99={p99_len} p95={p95_len}")
print(f"Samples above max_len={CFG['max_len']}: {too_long_count}/{len(df)}")

if too_long_count > 0:
    if CFG.get("filter_long_samples", False):
        print("Long samples will be dropped from each training fold (filter_long_samples=true).")
    else:
        print("Warning: long samples exist and will raise ValueError unless max_len is increased.")


In [ ]:
fold_results = []
for fold_id, (train_idx, val_idx) in enumerate(splits):
    result = train_one_fold(fold_id, train_idx, val_idx)
    fold_results.append(result)

    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

metrics_summary = [
    {
        "fold": r["fold"],
        "val_jaccard": r["val_jaccard"],
        "checkpoint": r["checkpoint"],
    }
    for r in fold_results
]
metrics_path = OUTPUT_DIR / "kfold_metrics.json"
metrics_path.write_text(json.dumps(metrics_summary, indent=2))
print("Saved metrics:", metrics_path)

history_rows = []
for r in fold_results:
    history_rows.extend(r.get("history", []))

history_df = pd.DataFrame(history_rows)
history_path = OUTPUT_DIR / "training_history.csv"
history_df.to_csv(history_path, index=False)
print("Saved training history:", history_path)

fold_results


In [ ]:
scores = [r["val_jaccard"] for r in fold_results]
print("Mean Jaccard:", float(np.mean(scores)))
print("Std Jaccard:", float(np.std(scores)))

if len(history_df) == 0:
    print("No training history points were logged. Reduce CFG['log_every'] to capture curves.")
else:
    fig, axes = plt.subplots(2, 2, figsize=(14, 9), sharex=True)
    metric_specs = [
        ("loss", "Total Loss"),
        ("ce", "CE Loss"),
        ("kl", "KL Loss"),
        ("sel", "Select Loss"),
    ]

    for ax, (col, title) in zip(axes.flat, metric_specs):
        for fold_id, fold_hist in history_df.groupby("fold"):
            ax.plot(fold_hist["global_step"], fold_hist[col], label=f"fold {int(fold_id)}")
        ax.set_title(title)
        ax.set_xlabel("Global Step")
        ax.set_ylabel(col)
        ax.grid(alpha=0.25)

    handles, labels = axes[0, 0].get_legend_handles_labels()
    if handles:
        fig.legend(handles, labels, loc="upper center", ncol=min(5, len(labels)))
    fig.suptitle("Training Loss Curves by Fold", y=1.02)
    plt.tight_layout()
    plt.show()


if len(history_df) > 0 and "grad_norm" in history_df.columns:
    fig, ax = plt.subplots(figsize=(10, 4))
    for fold_id, fold_hist in history_df.groupby("fold"):
        ax.plot(fold_hist["global_step"], fold_hist["grad_norm"], label=f"fold {int(fold_id)}")
    ax.set_title("Gradient Norm by Fold")
    ax.set_xlabel("Global Step")
    ax.set_ylabel("L2 Grad Norm")
    ax.grid(alpha=0.25)
    ax.legend(loc="best")
    plt.tight_layout()
    plt.show()

fig, ax = plt.subplots(figsize=(8, 4))
fold_ids = [int(r["fold"]) for r in fold_results]
fold_scores = [float(r["val_jaccard"]) for r in fold_results]
ax.bar(fold_ids, fold_scores)
ax.set_title("Validation Jaccard by Fold")
ax.set_xlabel("Fold")
ax.set_ylabel("Jaccard")
ax.set_ylim(0.0, max(1.0, max(fold_scores) + 0.05))
ax.grid(axis="y", alpha=0.25)
plt.show()
